# Intelligence Entropy: efficiency and breadth

Dario Cazzani · [AGI 2026 paper](https://doi.org/10.1007/978-3-032-33010-9_10) · [Repository](https://github.com/dariocazzani/intelligence-entropy)

Select **Runtime → Run all** in Colab. No local installation, training, credentials, or benchmark downloads are needed. The notebook uses the hosted runtime's NumPy and Matplotlib.

First we inspect the published sliding-puzzle ranking. Then we score an editable **illustrative** example using the experiment's context-selection rule. This is an explanation of the measurement, not a rerun of the 50-seed experiment.

## Load the existing measure

We fetch only `src/ie/measure.py` from the fixed repository commit `c0d99fe`, and check its SHA-256 before loading it. This reuses the existing NumPy-only implementation without installing the training stack. This cell needs internet access.

In [ ]:
import hashlib
import importlib.util
import tempfile
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

CORE_COMMIT = "c0d99fe15ac8bde2e361279769ca1432c5bb0d4f"
CORE_SHA256 = "129bed4611a7c5dec5a9ed180302a8e3516a31c98428ad0c381e08aeae1ad3f8"
core_url = (
    "https://raw.githubusercontent.com/dariocazzani/intelligence-entropy/"
    f"{CORE_COMMIT}/src/ie/measure.py"
)
with urlopen(core_url, timeout=30) as response:
    core_source = response.read()
if hashlib.sha256(core_source).hexdigest() != CORE_SHA256:
    raise RuntimeError("The downloaded measure does not match the pinned source.")

with tempfile.TemporaryDirectory() as module_dir:
    module_path = Path(module_dir) / "measure.py"
    module_path.write_bytes(core_source)
    spec = importlib.util.spec_from_file_location("ie_measure", module_path)
    measure = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(measure)

print(f"Loaded the measure from commit {CORE_COMMIT[:7]}.")

## Two factors, one score

$$I_{cd} = \eta H(C), \qquad \eta = -\Delta S / W$$

Here $\Delta S=S_{final}-S_{initial}$, so entropy reduction gives positive efficiency. $W$ is environmental action cost, not model FLOPs or runtime. $H(C)$ measures diversity under the evaluator's context-weighting rule, using natural logarithms (nats).

A system with high efficiency in just one context has $H(C)=0$. The following numbers are hypothetical, in arbitrary but fixed action-cost units.

In [ ]:
single_context = measure.i_cd(delta_s=-2.0, work=1.0, contexts=[0])
four_contexts = measure.i_cd(delta_s=-2.0, work=1.0, contexts=[0, 1, 2, 3])
print(f"Efficiency 2.0 in one context:   I_cd = {single_context:.3f}")
print(f"Efficiency 2.0 in four contexts: I_cd = {four_contexts:.3f}")
assert single_context == 0.0
assert np.isclose(four_contexts, 2.0 * np.log(4))

## The published sliding-puzzle comparison

The agents use the same greedy Manhattan-distance heuristic, restricted to different board widths. Outside the allowed widths they take random actions. Every agent is evaluated on all ten grid-width/scramble-depth contexts.

| Board | Scramble depths |
|---|---|
| 2×2 | 10 |
| 3×3 | 20, 50, 100 |
| 4×4 | 20, 50, 100 |
| 5×5 | 20, 50, 100 |

Scramble depth is the number of random legal moves from the solved board, not the shortest solution length. Environmental entropy is Shannon entropy over the histogram of tile Manhattan displacements. Each decision costs 1, including an invalid slide.

The **Easy** agent uses the heuristic on widths 2–3; **Full** uses it on widths 2–5. For each seed, the experiment retains contexts with net entropy reduction ($\eta_c>0$), averages their efficiencies equally, and assigns them uniform weight in $H(C)$. Thus $H(C)=\ln k$ for $k$ retained contexts. Positive efficiency does not imply a solved board.

The table below transcribes the rounded, 50-seed summaries in paper §4.2, Table 3. It contains no new experimental observations.

In [ ]:
# Entries are name, mean efficiency × 1000, mean H(C), mean score × 1000.
paper_rows = [
    ("Full", 0.781, 2.30, 1.795),
    ("Wide", 0.763, 1.94, 1.483),
    ("Hard", 0.702, 1.94, 1.355),
    ("Easy", 0.793, 1.38, 1.094),
    ("Narrow", 0.673, 0.33, 0.161),
    ("Random", 0.126, 0.07, 0.010),
]
table = [
    "| Agent | Mean η × 1000 | Mean H(C) | Mean I_cd × 1000 |",
    "|---|---:|---:|---:|",
]
table.extend(
    f"| {name} | {eta:.3f} | {diversity:.2f} | {score:.3f} |"
    for name, eta, diversity, score in paper_rows
)
display(Markdown("\n".join(table)))

In [ ]:
names = np.array([row[0] for row in paper_rows])
published_eta = np.array([row[1] for row in paper_rows])
published_score = np.array([row[3] for row in paper_rows])
highlights = {"Easy": "#d97706", "Full": "#0072b2"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
for ax, values, title, xlabel in zip(
    axes,
    (published_eta, published_score),
    ("Efficiency alone: Easy ranks first", "Combined score: Full ranks first"),
    ("Mean efficiency × 1000", "Mean I_cd × 1000"),
    strict=True,
):
    order = np.argsort(values)
    bars = ax.barh(
        names[order],
        values[order],
        color=[highlights.get(name, "#94a3b8") for name in names[order]],
    )
    ax.bar_label(bars, fmt="%.3f", padding=3)
    ax.set(title=title, xlabel=xlabel, xlim=(0, values.max() * 1.25))
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

assert names[np.argmax(published_eta)] == "Easy"
assert list(names[np.argsort(-published_score)]).index("Easy") == 3
assert names[np.argmax(published_score)] == "Full"

Easy's efficiency advantage is small, while Full demonstrates entropy reduction across more contexts. The combined ranking makes that breadth count. This does not guarantee that a generalist always wins; a sufficiently large efficiency advantage can outweigh a breadth disadvantage.

**Do not reconstruct exact mean scores by multiplying the table's mean factors.** The experiment computes a score within each seed, then averages those scores. In general, $\operatorname{mean}(\eta H)\ne\operatorname{mean}(\eta)\operatorname{mean}(H)$, and the printed values are rounded.

For a fresh reproduction, use [the experiment guide](https://github.com/dariocazzani/intelligence-entropy/blob/main/docs/reproducing-results.md#sliding-puzzle). It runs all six agents across 50 seeds.

## Edit an illustrative evaluation

Below are **invented per-context efficiencies**, not the observations behind the paper. Edit the ten numbers for each agent and rerun this section. All entries must refer to the same definitions of disorder, work, and contexts.

The context order is 2×2/10, 3×3/20, 3×3/50, 3×3/100, 4×4/20, 4×4/50, 4×4/100, 5×5/20, 5×5/50, 5×5/100. A negative number represents observed net entropy increase, not missing data.

For your own repeated episodes, first calculate each context's efficiency as $-\sum\Delta S/\sum W$ within one seed. Do not average episode ratios when work differs. Here we apply the puzzle's positive-efficiency gate, equally weight retained contexts, and use the repository's Shannon-diversity function.

In [ ]:
def score_contexts(eta_per_context: list[float]) -> tuple[float, float, float]:
    """Score one seed under the ten-context puzzle's uniform, gated protocol."""
    values = np.asarray(eta_per_context, dtype=float)
    if values.shape != (10,) or not np.isfinite(values).all():
        raise ValueError("Provide one finite efficiency for each of the ten contexts.")
    retained = np.flatnonzero(values > 0)
    if retained.size == 0:
        return 0.0, 0.0, 0.0
    eta = float(values[retained].mean())
    diversity = measure.context_diversity(retained)
    return eta, diversity, eta * diversity


illustrative_agents = {
    "Illustrative specialist": [0.0008] * 4 + [-0.0001] * 6,
    "Illustrative generalist": [0.00078] * 10,
}
for name, efficiencies in illustrative_agents.items():
    eta, diversity, score = score_contexts(efficiencies)
    print(f"{name}: η={eta:.6f}, H(C)={diversity:.3f}, I_cd={score:.6f}")

## Check the boundaries

No retained contexts gives zero; one gives zero diversity even when efficiency is positive. Invalid or missing context values must not silently turn into evidence of competence.

In [ ]:
assert score_contexts([0.0] * 10) == (0.0, 0.0, 0.0)
assert score_contexts([-0.1] * 10) == (0.0, 0.0, 0.0)
assert score_contexts([0.8] + [0.0] * 9) == (0.8, 0.0, 0.0)
np.testing.assert_allclose(score_contexts([0.8] * 4 + [0.0] * 6), (0.8, np.log(4), 0.8 * np.log(4)))
for invalid_values in ([0.1] * 9, [np.nan] * 10, [np.inf] * 10):
    try:
        score_contexts(invalid_values)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid context values were accepted.")
print("All boundary checks passed.")

## Before using your own results

Read [Applying the measure](https://github.com/dariocazzani/intelligence-entropy/blob/main/docs/applying-the-measure.md). State your disorder function, cost units, context partition, sampling budget, context weights, and any performance gate. Record failures and missing observations separately. Compute scores within each seed and then summarize them.

Scores can be compared within a fixed evaluation protocol. A puzzle score and an ARC compression score are not on a shared intelligence scale. Entropy reduction also needs a domain-specific interpretation: low Shannon entropy alone does not establish task success.

If you use this work, cite the [conference paper](https://doi.org/10.1007/978-3-032-33010-9_10). Author and publication metadata are in [CITATION.cff](https://github.com/dariocazzani/intelligence-entropy/blob/main/CITATION.cff).